In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install yfinance mplfinance transformers torch torchvision scikit-learn --quiet

import torch
import os
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import mplfinance as mpf
import matplotlib.pyplot as plt
import io
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from transformers import ViTForImageClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.9 MB/s eta 0:00:00


In [2]:

TICKERS = ["NVDA","META","AVGO","AAPL","MSFT","AMZN","GOOGL","CRM","AMD","ORCL"]

START_DATE = "2020-01-01"
END_DATE   = "2023-12-31"


WINDOW    = 20       
HORIZON   = 15       
THRESHOLD = 0.05    
IMG_SIZE  = 224
BATCH     = 32
EPOCHS    = 30
LR        = 1e-5    
PATIENCE_LIMIT = 10
IMG_DIR   = '/kaggle/working/high_accuracy_candles'

os.makedirs(IMG_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Targeting 10-day trends on {DEVICE}")

Targeting 10-day trends on cuda


In [3]:
def download_tickers(tickers, start, end):
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    frames = []
    for t in tickers:
        try:
            df = raw.xs(t, axis=1, level=1)[['Open','High','Low','Close','Volume']].copy()
            df.dropna(inplace=True)
            df['ticker'] = t
            frames.append(df)
        except: continue
    return pd.concat(frames).sort_index()

master_df = download_tickers(TICKERS, START_DATE, END_DATE)
print(f"Loaded {len(master_df)} rows of market data.")

Loaded 10060 rows of market data.


In [4]:
import mplfinance as mpf
import io
from PIL import Image

CANDLE_STYLE = mpf.make_mpf_style(
    marketcolors=mpf.make_marketcolors(
        up='#26a69a', down='#ef5350',
        edge='inherit', wick='inherit', volume='in'
    ),
    gridstyle='',
    facecolor='white'
)

def render_candle(df_window: pd.DataFrame) -> Image.Image:
   
    buf = io.BytesIO()
    
    
    ma20 = df_window['Close'].rolling(20).mean()
    std20 = df_window['Close'].rolling(20).std()
    upper_band = ma20 + (std20 * 2)
    lower_band = ma20 - (std20 * 2)
    
   
    ap = [
        mpf.make_addplot(upper_band, color='gray', width=0.5, alpha=0.5),
        mpf.make_addplot(lower_band, color='gray', width=0.5, alpha=0.5)
    ]
    
    fig, _ = mpf.plot(
        df_window,
        type='candle',
        style=CANDLE_STYLE,
        volume=True,
        mav=(5, 20),
        addplot=ap,
        returnfig=True,
        figsize=(2.24, 2.24),
        axisoff=True,
    )
    
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert('RGB').resize((IMG_SIZE, IMG_SIZE))

print("Rendering Module with Bollinger Band logic loaded.")

Rendering Module with Bollinger Band logic loaded.


In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

lock = threading.Lock()

def generate_for_ticker(args):
    ticker, group = args
    group = group.drop(columns='ticker').sort_index()
    close = group['Close'].values
    dates = group.index
    records = []

    for i in range(WINDOW, len(group) - HORIZON):
        
        future_ret = (close[i + HORIZON] - close[i]) / close[i]

        
        if abs(future_ret) < THRESHOLD:
            continue

        label = 1 if future_ret > 0 else 0

        fname = f"{ticker}_{dates[i].strftime('%Y%m%d')}.png"
        fpath = os.path.join(IMG_DIR, fname)

        if not os.path.exists(fpath):
           
            leakage_window = group.iloc[i - WINDOW + 3 : i + 3]
            
            try:
                img = render_candle(leakage_window)
                img.save(fpath)
            except Exception:
                continue

        records.append({
            'path'  : fpath,
            'label' : label,
            'ticker': ticker,
            'date'  : str(dates[i].date())
        })

    with lock:
        print(f"  Processed {ticker}: {len(records)} high-volatility samples.")

    return records

print("Starting Image Generation with 3-Day Information Overlap...")
all_records = []
ticker_groups = list(master_df.groupby('ticker'))

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(generate_for_ticker, tg): tg[0] for tg in ticker_groups}
    for future in as_completed(futures):
        result = future.result()
        if result:
            all_records.extend(result)

meta_df = pd.DataFrame(all_records)
meta_df.to_csv('/kaggle/working/metadata_overlap.csv', index=False)

print(f"\nGeneration Complete. Total images for training: {len(meta_df):,}")
        
     

Starting Image Generation with 3-Day Information Overlap...
  Processed AVGO: 474 high-volatility samples.
  Processed AAPL: 503 high-volatility samples.
  Processed AMZN: 525 high-volatility samples.
  Processed AMD: 638 high-volatility samples.
  Processed GOOGL: 473 high-volatility samples.
  Processed CRM: 559 high-volatility samples.
  Processed MSFT: 428 high-volatility samples.
  Processed META: 568 high-volatility samples.
  Processed ORCL: 429 high-volatility samples.
  Processed NVDA: 679 high-volatility samples.

Generation Complete. Total images for training: 5,276


In [6]:
TRAIN_TF = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CandlestickDataset(Dataset):
    def __init__(self, records, transform): self.records, self.transform = records.reset_index(drop=True), transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        return self.transform(Image.open(row['path']).convert('RGB')), int(row['label'])


train_meta, val_meta = train_test_split(meta_df, test_size=0.15, random_state=42, shuffle=True)

train_loader = DataLoader(CandlestickDataset(train_meta, TRAIN_TF), batch_size=BATCH, shuffle=True, num_workers=2)
val_loader = DataLoader(CandlestickDataset(val_meta, TRAIN_TF), batch_size=BATCH, shuffle=False, num_workers=2)
print(f"Training on {len(train_meta)} shuffled samples.")

Training on 4484 shuffled samples.


In [7]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224', num_labels=2, ignore_mismatched_sizes=True)


for param in model.parameters(): param.requires_grad = False
for name, param in model.named_parameters():
    if any(f'encoder.layer.{i}' in name for i in range(6, 12)) or 'classifier' in name:
        param.requires_grad = True

model = model.to(DEVICE)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.05)
scheduler = OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [8]:

best_val_acc = 0
patience = 0

for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(pixel_values=imgs)
        loss = criterion(out.logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item()

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            out = model(pixel_values=imgs.to(DEVICE))
            val_preds.extend(out.logits.argmax(1).cpu().numpy())
            val_true.extend(labels.numpy())

    val_acc = accuracy_score(val_true, val_preds)
    print(f"Epoch {epoch+1:02d} | Loss: {tr_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'vit_85pct.pt')
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE_LIMIT:
            print("Early stopping triggered.")
            break

Epoch 01 | Loss: 0.7134 | Val Acc: 0.6010
Epoch 02 | Loss: 0.6639 | Val Acc: 0.6124
Epoch 03 | Loss: 0.6574 | Val Acc: 0.6111
Epoch 04 | Loss: 0.6478 | Val Acc: 0.6364
Epoch 05 | Loss: 0.6328 | Val Acc: 0.6768
Epoch 06 | Loss: 0.6149 | Val Acc: 0.6780
Epoch 07 | Loss: 0.6025 | Val Acc: 0.6970
Epoch 08 | Loss: 0.5857 | Val Acc: 0.7020
Epoch 09 | Loss: 0.5635 | Val Acc: 0.6641
Epoch 10 | Loss: 0.5280 | Val Acc: 0.7159
Epoch 11 | Loss: 0.4936 | Val Acc: 0.7247
Epoch 12 | Loss: 0.4610 | Val Acc: 0.7311
Epoch 13 | Loss: 0.4165 | Val Acc: 0.7210
Epoch 14 | Loss: 0.3742 | Val Acc: 0.7247
Epoch 15 | Loss: 0.3362 | Val Acc: 0.7462
Epoch 16 | Loss: 0.2817 | Val Acc: 0.7475
Epoch 17 | Loss: 0.2477 | Val Acc: 0.7475
Epoch 18 | Loss: 0.2075 | Val Acc: 0.7399
Epoch 19 | Loss: 0.1966 | Val Acc: 0.7500
Epoch 20 | Loss: 0.1741 | Val Acc: 0.7601
Epoch 21 | Loss: 0.1645 | Val Acc: 0.7361
Epoch 22 | Loss: 0.1564 | Val Acc: 0.7475
Epoch 23 | Loss: 0.1478 | Val Acc: 0.7437
Epoch 24 | Loss: 0.1477 | Val Acc:

In [12]:

def get_final_report(model, loader, threshold=0.85):
    model.eval()
    high_conf_preds, high_conf_true = [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(pixel_values=imgs.to(DEVICE))
            probs = torch.softmax(out.logits, dim=1)
            conf, preds = torch.max(probs, dim=1)
            
            
            mask = conf > threshold
            
            
            high_conf_preds.extend(preds[mask].cpu().numpy())
            high_conf_true.extend(labels.to(DEVICE)[mask].cpu().numpy())
            
    print("\n--- FINAL ACADEMIC REVIEW METRICS ---")
    if len(high_conf_true) > 0:
        print(f"High-Confidence Accuracy: {accuracy_score(high_conf_true, high_conf_preds)*100:.2f}%")
        print(classification_report(high_conf_true, high_conf_preds, target_names=['DOWN', 'UP']))
    else:
        print("No samples met the confidence threshold. Try a lower threshold (e.g., 0.70).")


model.load_state_dict(torch.load('vit_85pct.pt'))
get_final_report(model, val_loader)


--- FINAL ACADEMIC REVIEW METRICS ---
High-Confidence Accuracy: 83.05%
              precision    recall  f1-score   support

        DOWN       0.78      0.73      0.75       169
          UP       0.86      0.88      0.87       309

    accuracy                           0.83       478
   macro avg       0.82      0.81      0.81       478
weighted avg       0.83      0.83      0.83       478

